# ST-GCN joint_bone - multi-seed baseline, with the sampler fix

Two problems from the last notebook, both addressed here:

1. **`balanced_sampler` was built once and reused by reference across all three training runs in a loop.** If its internal RNG state keeps advancing across uses rather than resetting per call, later runs in the loop weren't actually starting from a clean seeded state - likely the remaining source of the 72.63% vs 71.10% swing on a "same config" run. Fixed here: the sampler is now built fresh, inside the training function, per call.

2. **Single-run numbers can't be trusted given the noise floor we just measured** (>1.5pp swing even under controlled seeding). This trains the best known config - `joint_bone`, `weight_decay=1e-3`, `step_size=10` (the schedule that actually worked, not the stretched ones that just caused more overfitting) - across 3 different seeds, and reports the real mean and standard deviation instead of a single best-run peak.

This is slower than a single run but it's the only way to know whether ~71-72% top-1 is a genuine, stable result or something that got lucky once.

### determinism - before torch touches CUDA

In [1]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import sys
import json
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, top_k_accuracy_score,
    confusion_matrix, precision_recall_fscore_support,
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

sys.path.insert(0, "/workspace/badminton-honours-project")
import dataset as ds

import mmaction.models  # noqa: F401
from mmaction.utils import register_all_modules
register_all_modules()
from mmaction.registry import MODELS

from clearml import Task

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("device:", device)


device: cuda:0


### DataLoader worker-seeding fix - same as before

In [2]:
def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


### paths

In [3]:
OUT_ROOT = "/workspace/outputs/stgcn_jointbone_multiseed"
CKPT_ROOT = "/workspace/checkpoints/stgcn_jointbone_multiseed"
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(CKPT_ROOT, exist_ok=True)


### data - built once, but NOT the sampler

The datasets themselves (`train_ds`, `val_ds`, `test_ds`) are safe to build once and reuse - they don't carry per-call RNG state the way a sampler might. `balanced_sampler` is deliberately NOT built here - it's constructed fresh inside `train_one_seed()` instead.

In [4]:
manifest = ds.load_filtered_manifest(
    "/workspace/data/split_manifest.csv",
    "/workspace/data/extraction_log.csv",
)

class_names = sorted(manifest["class_name"].unique())
class_to_id = {name: i for i, name in enumerate(class_names)}
num_classes = len(class_names)

train_manifest = manifest[manifest["split"] == "train"].reset_index(drop=True)
val_manifest = manifest[manifest["split"] == "val"].reset_index(drop=True)
test_manifest = manifest[manifest["split"] == "test"].reset_index(drop=True)

KEYPOINTS_ROOT = "/workspace/data/keypoints_smoothed"

train_ds = ds.BadmintonPoseDataset(train_manifest, KEYPOINTS_ROOT, class_to_id, augmentation_policy="advanced")
val_ds = ds.BadmintonPoseDataset(val_manifest, KEYPOINTS_ROOT, class_to_id)
test_ds = ds.BadmintonPoseDataset(test_manifest, KEYPOINTS_ROOT, class_to_id)

val_loader = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

print(f"{num_classes} classes, train {len(train_manifest)}, val {len(val_ds)}, test {len(test_ds)}")


dropped 4 manually-excluded clips from the manifest (7814 remain)
18 classes, train 6251, val 781, test 782


### batch -> ST-GCN input - joint_bone, in_channels=5

In [5]:
def batch_to_stgcn_input(batch):
    x = torch.cat([batch["keypoints"].float(), batch["bone"].float()], dim=-1)
    return x.unsqueeze(1).unsqueeze(1)


### SMOKE TEST - run alone first

In [6]:
smoke_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
smoke_batch = next(iter(smoke_loader))
smoke_input = batch_to_stgcn_input(smoke_batch)
print("input shape:", smoke_input.shape)
assert smoke_input.shape[-1] == 5, f"expected 5 channels, got {smoke_input.shape[-1]}"
print("smoke test passed")


input shape: torch.Size([4, 1, 1, 30, 17, 5])
smoke test passed


### metrics

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    all_labels, all_preds, all_probs = [], [], []

    for batch in loader:
        x = batch_to_stgcn_input(batch).to(device)
        y = batch["label"].to(device)
        logits = model(x, mode="tensor", stage="head")
        probs = torch.softmax(logits, dim=1)

        all_labels.append(y.cpu().numpy())
        all_preds.append(probs.argmax(dim=1).cpu().numpy())
        all_probs.append(probs.cpu().numpy())

    labels = np.concatenate(all_labels)
    preds = np.concatenate(all_preds)
    probs = np.concatenate(all_probs)

    return {
        "top1": accuracy_score(labels, preds),
        "top5": top_k_accuracy_score(labels, probs, k=5, labels=list(range(num_classes))),
        "mca": balanced_accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "labels": labels, "preds": preds,
    }


### the reusable training function - sampler now built fresh per call

The key fix versus last time: `ds.build_class_balanced_sampler(...)` is called inside this function, using the same `seed` as everything else, instead of being built once outside and shared across calls.

In [8]:
def train_one_seed(seed, step_size=10, num_epochs=60, patience=20, batch_size=16,
                    lr=0.01, weight_decay=1e-3):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)

    # sampler built fresh here, seeded the same as everything else - the actual fix
    balanced_sampler = ds.build_class_balanced_sampler(train_manifest, target_per_class=500, seed=seed)

    model_cfg = dict(
        type="RecognizerGCN",
        backbone=dict(
            type="STGCN",
            graph_cfg=dict(layout="coco", mode="stgcn_spatial"),
            in_channels=5, base_channels=64, ch_ratio=2,
            num_stages=9, inflate_stages=[4, 7], down_stages=[4, 7],
        ),
        cls_head=dict(type="GCNHead", num_classes=num_classes, in_channels=256,
                      loss_cls=dict(type="CrossEntropyLoss")),
    )
    model = MODELS.build(model_cfg).to(device)

    loader_generator = torch.Generator()
    loader_generator.manual_seed(seed)

    train_loader = DataLoader(
        train_ds, batch_size=batch_size, sampler=balanced_sampler,
        num_workers=4, pin_memory=True, persistent_workers=True,
        worker_init_fn=seed_worker, generator=loader_generator,
    )

    config_name = f"seed{seed}"
    task = Task.init(project_name="BadmintonPoseAnalysis", task_name=f"stgcn_jointbone_multiseed_{config_name}", reuse_last_task_id=False)
    logger = task.get_logger()
    task.connect({
        "stream": "joint_bone", "seed": seed, "step_size": step_size,
        "lr": lr, "weight_decay": weight_decay, "num_epochs": num_epochs,
        "batch_size": batch_size, "augmentation_policy": "advanced",
        "sampler_fix": "built fresh per call, not shared across a loop",
    })

    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=step_size, gamma=0.1)

    best_val_top1 = 0.0
    best_epoch = -1
    epochs_without_improvement = 0
    best_state = None

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_labels, epoch_preds = [], []

        for batch in train_loader:
            x = batch_to_stgcn_input(batch).to(device)
            y = batch["label"].to(device)

            optimizer.zero_grad()
            logits = model(x, mode="tensor", stage="head")
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item() * x.size(0)
            epoch_labels.append(y.cpu().numpy())
            epoch_preds.append(logits.argmax(dim=1).cpu().numpy())

        scheduler.step()

        train_loss = epoch_loss / len(train_loader.dataset)
        train_acc = accuracy_score(np.concatenate(epoch_labels), np.concatenate(epoch_preds))
        val_metrics = evaluate(model, val_loader)

        logger.report_scalar("loss", "train", train_loss, iteration=epoch)
        logger.report_scalar("accuracy", "train", train_acc, iteration=epoch)
        logger.report_scalar("accuracy", "val_top1", val_metrics["top1"], iteration=epoch)
        logger.report_scalar("accuracy", "val_mca", val_metrics["mca"], iteration=epoch)

        if val_metrics["top1"] > best_val_top1:
            best_val_top1 = val_metrics["top1"]
            best_epoch = epoch
            epochs_without_improvement = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1

        if epoch % 10 == 0 or epoch == num_epochs - 1:
            print(f"[{config_name}] epoch {epoch:3d}  train_acc={train_acc:.4f}  "
                  f"val_top1={val_metrics['top1']:.4f}  val_mca={val_metrics['mca']:.4f}")

        if epochs_without_improvement >= patience:
            print(f"[{config_name}] early stopping at epoch {epoch}, best val_top1={best_val_top1:.4f} (epoch {best_epoch})")
            break

    model.load_state_dict(best_state)

    train_final = evaluate(model, DataLoader(train_ds, batch_size=64, shuffle=False, num_workers=4, pin_memory=True))
    test_final = evaluate(model, test_loader)

    metrics = {
        "config_name": config_name,
        "seed": seed,
        "best_epoch": best_epoch,
        "train_acc": train_final["top1"],
        "test_top1": test_final["top1"],
        "test_top5": test_final["top5"],
        "test_mca": test_final["mca"],
        "test_macro_f1": test_final["macro_f1"],
        "train_test_gap": train_final["top1"] - test_final["top1"],
    }

    for metric_name in ["train_acc", "test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap", "best_epoch"]:
        logger.report_single_value(name=metric_name, value=metrics[metric_name])

    cm = confusion_matrix(test_final["labels"], test_final["preds"], labels=list(range(num_classes)))
    logger.report_confusion_matrix(
        title="Test Confusion Matrix", series=config_name,
        matrix=cm, xaxis="Predicted", yaxis="Actual",
        xlabels=class_names, ylabels=class_names,
    )

    precision, recall, f1, support = precision_recall_fscore_support(
        test_final["labels"], test_final["preds"], labels=list(range(num_classes)), zero_division=0,
    )
    per_class_df = pd.DataFrame({
        "class": class_names, "precision": precision, "recall": recall,
        "f1": f1, "support": support,
    })
    logger.report_table(title="Per-Class Test Metrics", series=config_name, table_plot=per_class_df)

    ckpt_path = os.path.join(CKPT_ROOT, f"{config_name}_best.pt")
    torch.save(model.state_dict(), ckpt_path)
    task.upload_artifact("checkpoint", ckpt_path)
    per_class_df.to_csv(os.path.join(OUT_ROOT, f"per_class_{config_name}.csv"), index=False)
    task.close()

    return metrics


### run three seeds

3x a single run. Same cost as every multi-config comparison today, but this time all three runs share the exact same config - only the seed differs - so the spread between them directly measures the real noise floor rather than comparing different ideas.

In [9]:
seeds_to_test = [42, 123, 7]

all_results = {}

for seed in seeds_to_test:
    print(f"\n=== training seed={seed} ===")
    metrics = train_one_seed(seed)
    all_results[metrics["config_name"]] = metrics
    print(f"[{metrics['config_name']}] FINAL: test_top1={metrics['test_top1']:.4f}  "
          f"gap={metrics['train_test_gap']:.4f}  test_mca={metrics['test_mca']:.4f}  "
          f"test_macro_f1={metrics['test_macro_f1']:.4f}")



=== training seed=42 ===
ClearML Task: created new task id=35a43d4bfee84392bce79216f97a69a9
2026-09-10 11:22:27,923 - clearml.Repository Detection - WARNING - Failed accessing the jupyter server(s): []
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/35a43d4bfee84392bce79216f97a69a9/output/log
[seed42] epoch   0  train_acc=0.2898  val_top1=0.5506  val_mca=0.3714
[seed42] epoch  10  train_acc=0.7645  val_top1=0.6761  val_mca=0.6453
[seed42] epoch  20  train_acc=0.8910  val_top1=0.7093  val_mca=0.6425
[seed42] epoch  30  train_acc=0.9142  val_top1=0.7196  val_mca=0.6293
[seed42] epoch  40  train_acc=0.9167  val_top1=0.7196  val_mca=0.6348
[seed42] early stopping at epoch 47, best val_top1=0.7324 (epoch 27)


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  7.28MB/s]: 


[seed42] FINAL: test_top1=0.7110  gap=0.2161  test_mca=0.6160  test_macro_f1=0.6302

=== training seed=123 ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=f142b469fe274c359e69f2aef2a32a53
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/f142b469fe274c359e69f2aef2a32a53/output/log
[seed123] epoch   0  train_acc=0.2997  val_top1=0.5224  val_mca=0.4250
[seed123] epoch  10  train_acc=0.7698  val_top1=0.6901  val_mca=0.6467
[seed123] epoch  20  train_acc=0.8890  val_top1=0.7234  val_mca=0.6542
[seed123] epoch  30  train_acc=0.9192  val_top1=0.7311  val_mca=0.6497
[seed123] epoch  40  train_acc=0.9250  val_top1=0.7196  val_mca=0.6414
[seed123] epoch  50  train_acc=0.9225  val_top1=0.7119  val_mca=0.6256
[seed123] early stopping at epoch 54, best val_top1=0.7350 (epoch 34)


...

[seed123] FINAL: test_top1=0.7161  gap=0.2173  test_mca=0.6371  test_macro_f1=0.6454

=== training seed=7 ===


Failed accessing the jupyter server(s): []


ClearML Task: created new task id=e8227e809dfb483ebd5106cd248a00ba
ClearML results page: https://app.clear.ml/projects/89838577f7964905a3b492353933fcb2/tasks/e8227e809dfb483ebd5106cd248a00ba/output/log
[seed7] epoch   0  train_acc=0.3074  val_top1=0.5314  val_mca=0.4366
[seed7] epoch  10  train_acc=0.7881  val_top1=0.6927  val_mca=0.6433
[seed7] epoch  20  train_acc=0.9159  val_top1=0.7298  val_mca=0.6418
[seed7] epoch  30  train_acc=0.9454  val_top1=0.7260  val_mca=0.6346
[seed7] epoch  40  train_acc=0.9429  val_top1=0.7350  val_mca=0.6412
[seed7] epoch  50  train_acc=0.9430  val_top1=0.7452  val_mca=0.6547
[seed7] epoch  59  train_acc=0.9447  val_top1=0.7362  val_mca=0.6450


███████████████████████████████ 100% | 11.66/11.66 MB [00:01<00:00,  7.26MB/s]: 


[seed7] FINAL: test_top1=0.6995  gap=0.2608  test_mca=0.6136  test_macro_f1=0.6264


### the real result - mean +/- std across seeds, not a single peak

In [10]:
results_df = pd.DataFrame(list(all_results.values()))

summary_stats = results_df[["test_top1", "test_top5", "test_mca", "test_macro_f1", "train_test_gap"]].agg(["mean", "std", "min", "max"])

LI_ET_AL_BENCHMARK = {"test_top1": 0.7441, "test_top5": 0.9376, "test_mca": 0.6144}

print("per-seed results:")
print(results_df[["seed", "test_top1", "test_mca", "test_macro_f1", "train_test_gap"]].to_string(index=False))
print()
print("summary (mean +/- std across 3 seeds):")
print(summary_stats)
print()
print(f"Li et al. benchmark: test_top1={LI_ET_AL_BENCHMARK['test_top1']:.4f}  "
      f"test_top5={LI_ET_AL_BENCHMARK['test_top5']:.4f}  test_mca={LI_ET_AL_BENCHMARK['test_mca']:.4f}")

results_df.to_csv(os.path.join(OUT_ROOT, "per_seed_results.csv"), index=False)
summary_stats.to_csv(os.path.join(OUT_ROOT, "summary_stats.csv"))


per-seed results:
 seed  test_top1  test_mca  test_macro_f1  train_test_gap
   42   0.710997  0.615995       0.630229        0.216054
  123   0.716113  0.637108       0.645396        0.217338
    7   0.699488  0.613637       0.626355        0.260838

summary (mean +/- std across 3 seeds):
      test_top1  test_top5  test_mca  test_macro_f1  train_test_gap
mean   0.708866   0.944160  0.622247       0.633993        0.231410
std    0.008514   0.000738  0.012924       0.010063        0.025493
min    0.699488   0.943734  0.613637       0.626355        0.216054
max    0.716113   0.945013  0.637108       0.645396        0.260838

Li et al. benchmark: test_top1=0.7441  test_top5=0.9376  test_mca=0.6144


---

The `std` row is the real noise floor for this config - report `mean +/- std` in the thesis for this result, not a single best-run number. If `std` for test_top1 is small (well under 1pp), today's earlier single-run comparisons were probably reasonably trustworthy after all. If it's large (like the ~1.5pp swing that motivated this notebook), that's honest, important context: most of today's config-to-config differences likely weren't distinguishable from pure noise, and the fair comparison against Li et al.'s benchmark should use this mean, not whichever single run happened to score highest.